### Load & Parse Data

In [1]:
import pandas as pd
import numpy as np

telemetry = pd.read_csv('D:/Projects/predictiq/data/PdM_telemetry.csv')
errors     = pd.read_csv('D:/Projects/predictiq/data/PdM_errors.csv')
failures   = pd.read_csv('D:/Projects/predictiq/data/PdM_failures.csv')
machines   = pd.read_csv('D:/Projects/predictiq/data/PdM_machines.csv')
maint      = pd.read_csv('D:/Projects/predictiq/data/PdM_maint.csv')

for df in [telemetry, errors, failures, maint]:
    df['datetime'] = pd.to_datetime(df['datetime'])

print("All files loaded.")

All files loaded.


### Rolling Telemetry Features (3h and 24h windows)

In [2]:
sensors = ['volt', 'rotate', 'pressure', 'vibration']

telemetry = telemetry.sort_values(['machineID', 'datetime'])

# Rolling mean and std over 3h and 24h windows
for window in [3, 24]:
    for sensor in sensors:
        telemetry[f'{sensor}_mean_{window}h'] = (
            telemetry.groupby('machineID')[sensor]
            .transform(lambda x: x.rolling(window, min_periods=1).mean())
        )
        telemetry[f'{sensor}_std_{window}h'] = (
            telemetry.groupby('machineID')[sensor]
            .transform(lambda x: x.rolling(window, min_periods=1).std().fillna(0))
        )

print(f"Telemetry shape after rolling features: {telemetry.shape}")
print(f"New columns added: {[c for c in telemetry.columns if 'mean' in c or 'std' in c]}")

Telemetry shape after rolling features: (876100, 22)
New columns added: ['volt_mean_3h', 'volt_std_3h', 'rotate_mean_3h', 'rotate_std_3h', 'pressure_mean_3h', 'pressure_std_3h', 'vibration_mean_3h', 'vibration_std_3h', 'volt_mean_24h', 'volt_std_24h', 'rotate_mean_24h', 'rotate_std_24h', 'pressure_mean_24h', 'pressure_std_24h', 'vibration_mean_24h', 'vibration_std_24h']


### Error Count Features

In [3]:
# One-hot encode error types
errors_dummies = pd.get_dummies(errors, columns=['errorID'], prefix='error')
error_cols = [c for c in errors_dummies.columns if c.startswith('error_')]

# Merge errors onto telemetry timeline
telemetry_with_errors = telemetry[['datetime', 'machineID']].copy()

# For each hour in telemetry, count errors in past 24h per machine
errors_dummies = errors_dummies.sort_values(['machineID', 'datetime'])
telemetry_with_errors = telemetry_with_errors.sort_values(['machineID', 'datetime'])

# Use merge_asof per machine to get rolling error counts
error_features = []

for machine_id in telemetry['machineID'].unique():
    tel_m = telemetry_with_errors[telemetry_with_errors['machineID'] == machine_id].copy()
    err_m = errors_dummies[errors_dummies['machineID'] == machine_id].copy()
    
    for col in error_cols:
        tel_m[col] = 0
    
    for _, err_row in err_m.iterrows():
        # Mark errors within 24h window
        mask = (tel_m['datetime'] >= err_row['datetime']) & \
               (tel_m['datetime'] < err_row['datetime'] + pd.Timedelta(hours=24))
        for col in error_cols:
            tel_m.loc[mask, col] += err_row[col]
    
    error_features.append(tel_m)

error_features_df = pd.concat(error_features).sort_values(['machineID', 'datetime'])
telemetry = telemetry.merge(
    error_features_df[['datetime', 'machineID'] + error_cols],
    on=['datetime', 'machineID'],
    how='left'
)
telemetry[error_cols] = telemetry[error_cols].fillna(0)

print(f"Error features added: {error_cols}")
print(f"Telemetry shape: {telemetry.shape}")

Error features added: ['error_error1', 'error_error2', 'error_error3', 'error_error4', 'error_error5']
Telemetry shape: (876100, 27)


### Days Since Last Maintenance

In [4]:
comp_list = maint['comp'].unique()

for comp in comp_list:
    # Get last maintenance date per machine per comp up to each telemetry timestamp
    maint_comp = maint[maint['comp'] == comp][['machineID', 'datetime']].copy()
    maint_comp = maint_comp.rename(columns={'datetime': 'maint_datetime'})
    
    # Cross join telemetry timestamps with maintenance records per machine
    merged = telemetry[['machineID', 'datetime']].merge(maint_comp, on='machineID', how='left')
    
    # Only keep maintenance events that happened BEFORE or AT the telemetry timestamp
    merged = merged[merged['maint_datetime'] <= merged['datetime']]
    
    # Keep only the most recent maintenance event before each telemetry timestamp
    merged = merged.sort_values('maint_datetime').groupby(
        ['machineID', 'datetime'], as_index=False
    ).last()
    
    # Calculate days since last maintenance
    merged[f'days_since_{comp}'] = (
        merged['datetime'] - merged['maint_datetime']
    ).dt.total_seconds() / 86400
    
    # Merge back into telemetry
    telemetry = telemetry.merge(
        merged[['machineID', 'datetime', f'days_since_{comp}']],
        on=['machineID', 'datetime'],
        how='left'
    )
    
    # Fill NaN — machine had no prior maintenance recorded
    telemetry[f'days_since_{comp}'] = telemetry[f'days_since_{comp}'].fillna(9999)

days_since_cols = [f'days_since_{c}' for c in comp_list]
print(f"Maintenance features added: {days_since_cols}")
print(f"Telemetry shape: {telemetry.shape}")

Maintenance features added: ['days_since_comp2', 'days_since_comp4', 'days_since_comp3', 'days_since_comp1']
Telemetry shape: (876100, 31)


### Merge Machine Metadata

In [5]:
telemetry = telemetry.merge(machines, on='machineID', how='left')

# One-hot encode model
telemetry = pd.get_dummies(telemetry, columns=['model'], prefix='model')

print(f"After merging machine metadata: {telemetry.shape}")
print(f"Model columns: {[c for c in telemetry.columns if c.startswith('model_')]}")

After merging machine metadata: (876100, 36)
Model columns: ['model_model1', 'model_model2', 'model_model3', 'model_model4']


### Build Target Label (Failure in next 24h)

In [6]:
# Label each row: did this machine fail within the next 24 hours?
telemetry['label'] = 0

for _, failure_row in failures.iterrows():
    machine = failure_row['machineID']
    fail_time = failure_row['datetime']
    
    # Mark 24h window before failure as positive
    mask = (
        (telemetry['machineID'] == machine) &
        (telemetry['datetime'] >= fail_time - pd.Timedelta(hours=24)) &
        (telemetry['datetime'] < fail_time)
    )
    telemetry.loc[mask, 'label'] = 1

print(f"Label distribution:")
print(telemetry['label'].value_counts())
print(f"\nFailure rate: {telemetry['label'].mean()*100:.2f}%")

Label distribution:
label
0    858916
1     17184
Name: count, dtype: int64

Failure rate: 1.96%


### Save Master DataFrame

In [7]:
# Drop rows with NaN (early rows before rolling windows stabilize)
telemetry.dropna(inplace=True)

# Save
telemetry.to_csv('D:/Projects/predictiq/data/master_features.csv', index=False)

print(f"Master dataframe saved.")
print(f"Final shape: {telemetry.shape}")
print(f"Features: {telemetry.shape[1] - 3} (excluding datetime, machineID, label)")
print(f"\nAll columns:")
for col in telemetry.columns:
    print(f"  {col}")

Master dataframe saved.
Final shape: (876100, 37)
Features: 34 (excluding datetime, machineID, label)

All columns:
  datetime
  machineID
  volt
  rotate
  pressure
  vibration
  volt_mean_3h
  volt_std_3h
  rotate_mean_3h
  rotate_std_3h
  pressure_mean_3h
  pressure_std_3h
  vibration_mean_3h
  vibration_std_3h
  volt_mean_24h
  volt_std_24h
  rotate_mean_24h
  rotate_std_24h
  pressure_mean_24h
  pressure_std_24h
  vibration_mean_24h
  vibration_std_24h
  error_error1
  error_error2
  error_error3
  error_error4
  error_error5
  days_since_comp2
  days_since_comp4
  days_since_comp3
  days_since_comp1
  age
  model_model1
  model_model2
  model_model3
  model_model4
  label
